# Story Prototype: Which Investment Won?

This notebook keeps calculation code minimal. It loads two price files, calls the engine functions, then focuses on the visual story.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-cache")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from analysis_engine.calculations import compare_investments
from analysis_engine.data_access import load_prices
from analysis_engine.utils import money, row_on_or_after, row_on_or_before

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)


## Inputs

Change these to try another story. The notebook chooses the latest shared market date and looks back the selected number of years.

In [ ]:
STOCK_A = "AAPL"
STOCK_B = "MSFT"
INITIAL_AMOUNT = 10_000
YEARS = 10

# Data loading and formatting helpers are imported from analysis_engine.


In [ ]:
prices_a = load_prices(STOCK_A)
prices_b = load_prices(STOCK_B)
end_date = min(prices_a["date"].max(), prices_b["date"].max())
target_start = end_date - pd.DateOffset(years=YEARS)

start_a = row_on_or_after(prices_a, target_start)
end_a = row_on_or_before(prices_a, end_date)
start_b = row_on_or_after(prices_b, target_start)
end_b = row_on_or_before(prices_b, end_date)

comparison = compare_investments(
    STOCK_A,
    start_a["adj_close"],
    end_a["adj_close"],
    STOCK_B,
    start_b["adj_close"],
    end_b["adj_close"],
    INITIAL_AMOUNT,
    years=YEARS,
)

comparison


## The Headline

This is the sentence the app can put above the chart.

In [ ]:
print(
    f"Over the last {YEARS} years, {comparison.winner} beat {comparison.loser} "
    f"by {money(comparison.final_value_difference)} on a {money(INITIAL_AMOUNT)} investment. "
    f"That is a {comparison.winner_multiple:,.2f}x advantage."
)


## Investment Path

The main comparison should show the journey, not only the final winner.

In [ ]:
def wealth_path(df, start_row, end_dt):
    period = df[(df["date"] >= start_row["date"]) & (df["date"] <= end_dt)].copy()
    period["wealth"] = INITIAL_AMOUNT * period["adj_close"] / start_row["adj_close"]
    return period


path_a = wealth_path(prices_a, start_a, end_date)
path_b = wealth_path(prices_b, start_b, end_date)

fig, ax = plt.subplots(figsize=(11, 5.8))
ax.plot(path_a["date"], path_a["wealth"], label=STOCK_A, color="#2563eb", linewidth=2.7)
ax.plot(path_b["date"], path_b["wealth"], label=STOCK_B, color="#0f766e", linewidth=2.7)

for ticker, result, color in [(STOCK_A, comparison.first, "#2563eb"), (STOCK_B, comparison.second, "#0f766e")]:
    ax.scatter(end_date, result.final_value, color=color, s=70, zorder=3)
    ax.annotate(
        f"{ticker}: {money(result.final_value)}",
        xy=(end_date, result.final_value),
        xytext=(-132, 24 if ticker == STOCK_A else -36),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": color},
        fontsize=10,
        color=color,
    )

ax.set_title(f"{money(INITIAL_AMOUNT)} invested in {STOCK_A} vs {STOCK_B}", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Portfolio value")
ax.yaxis.set_major_formatter(lambda value, pos: money(value))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## Final Scorecard

A compact scorecard gives the frontend card layout something concrete to copy.

In [ ]:
scorecard = pd.DataFrame([
    {
        "ticker": comparison.first.ticker,
        "final_value": comparison.first.final_value,
        "profit": comparison.first.profit,
        "total_return": comparison.first.total_return,
        "annualized_return": comparison.first.annualized_return,
    },
    {
        "ticker": comparison.second.ticker,
        "final_value": comparison.second.final_value,
        "profit": comparison.second.profit,
        "total_return": comparison.second.total_return,
        "annualized_return": comparison.second.annualized_return,
    },
])

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#2563eb" if ticker == comparison.winner else "#94a3b8" for ticker in scorecard["ticker"]]
ax.bar(scorecard["ticker"], scorecard["final_value"], color=colors, width=0.52)
for _, row in scorecard.iterrows():
    ax.text(row["ticker"], row["final_value"], money(row["final_value"]), ha="center", va="bottom", fontsize=11)
ax.set_title("Final value after the selected period", loc="left", fontsize=15, fontweight="bold")
ax.set_ylabel("Final value")
ax.yaxis.set_major_formatter(lambda value, pos: money(value))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

scorecard
